# Track A — Fase 4: Generate folds_v3.csv
**BDC Satria Data 2026 — Klasifikasi Citra Sampah**

Notebook ini menghasilkan `folds_v3.csv` dengan menerapkan **dua cleaning log** ke `folds_v2.csv`:

| Cleaning Log | Sumber | Keterangan |
|---|---|---|
| `cleaning_log_terisi.csv` | Review manual (raka) | Data noisy / ambigu yang sudah dipilih secara kualitatif |
| `cleaning_log_misclassified.csv` | Misclassified SO400M+KNN (aga) | Gambar yang salah prediksi model, diputuskan keep/drop/relabel |

**Strategi:** Tidak re-split ulang. Fold assignment dari `folds_v2.csv` dipertahankan.
Hanya baris yang di-drop dihapus, dan label yang di-relabel diganti.

> **Prasyarat:** `folds_v2.csv`, `cleaning_log_terisi.csv`, `cleaning_log_misclassified.csv` sudah tersedia di Drive.

---

In [ ]:
# ─── Cell 1: Setup ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
result = subprocess.run(
    ['git', 'clone', 'https://github.com/agaggigit/satria-data-bdcugm02.git', '/content/repo'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
sys.path.insert(0, '/content/repo')
print('✅ Setup selesai')

In [ ]:
# ─── Cell 2: Konfigurasi Path ─────────────────────────────────────────────────
# ⚠️ SESUAIKAN path di bawah jika perlu!

DRIVE_BASE              = '/content/drive/MyDrive/BDC2026apace'
FOLDS_V2_CSV            = f'{DRIVE_BASE}/output_trackA/folds_v2.csv'
CLEANING_LOG_TERISI     = f'{DRIVE_BASE}/output_trackA/cleaning_log_terisi.csv'
CLEANING_LOG_MISCLS     = f'{DRIVE_BASE}/output_trackA/cleaning_log_misclassified.csv'
OUTPUT_DIR              = f'{DRIVE_BASE}/output_trackA'

import os
for name, path in [
    ('folds_v2.csv',                    FOLDS_V2_CSV),
    ('cleaning_log_terisi.csv',         CLEANING_LOG_TERISI),
    ('cleaning_log_misclassified.csv',  CLEANING_LOG_MISCLS),
]:
    status = '✅' if os.path.exists(path) else '❌ TIDAK ADA'
    print(f'  {status} {name}: {path}')

In [ ]:
# ─── Cell 3: Load & Normalise Cleaning Logs ───────────────────────────────────
import pandas as pd

# --- Load folds_v2 (baseline) ---
folds_v2 = pd.read_csv(FOLDS_V2_CSV)
print(f'folds_v2 loaded : {len(folds_v2):,} baris | kolom: {list(folds_v2.columns)}')

# --- Load cleaning log terisi (review manual) ---
log_terisi = pd.read_csv(CLEANING_LOG_TERISI)
log_terisi['keputusan'] = log_terisi['keputusan'].str.strip().str.lower()
print(f'\ncleaning_log_terisi  : {len(log_terisi):,} baris')
print(log_terisi['keputusan'].value_counts())

# --- Load cleaning log misclassified (review aga) ---
log_miscls = pd.read_csv(CLEANING_LOG_MISCLS)
log_miscls['keputusan'] = log_miscls['keputusan'].str.strip().str.lower()
print(f'\ncleaning_log_misclassified : {len(log_miscls):,} baris')
print(log_miscls['keputusan'].value_counts())

# --- Standarisasi kolom agar bisa digabung ---
# Keduanya harus punya: filepath, keputusan, label_baru
COLS_NEEDED = ['filepath', 'keputusan', 'label_baru']

log_terisi_std  = log_terisi[COLS_NEEDED].copy()
log_miscls_std  = log_miscls[COLS_NEEDED].copy()

# Gabung (concat, lalu dedup — prioritaskan log_terisi jika ada konflik)
log_all = pd.concat([log_terisi_std, log_miscls_std], ignore_index=True)
log_all['filepath'] = log_all['filepath'].str.strip().str.replace('"', '', regex=False)
folds_v2['filepath'] = folds_v2['filepath'].str.strip()

# Hapus duplikat filepath — keep first (terisi punya prioritas karena concat duluan)
before_dedup = len(log_all)
log_all = log_all.drop_duplicates(subset='filepath', keep='first')
print(f'\nGabungan log     : {before_dedup} → {len(log_all)} (setelah dedup)')

# Breakdown final
print('\nRekapitulasi keputusan:')
print(log_all['keputusan'].value_counts())

In [ ]:
# ─── Cell 4: Cek Coverage ────────────────────────────────────────────────────
# Seberapa banyak filepath di log yang ada di folds_v2?

fps_v2    = set(folds_v2['filepath'])
fps_log   = set(log_all['filepath'])

in_v2     = fps_log & fps_v2
not_in_v2 = fps_log - fps_v2

print(f'Filepath di log yang cocok dengan folds_v2 : {len(in_v2):,}')
print(f'Filepath di log yang TIDAK ada di folds_v2 : {len(not_in_v2):,}')

if not_in_v2:
    print('\n⚠️ Filepath ini ada di log tapi tidak ada di folds_v2 (akan di-skip):')
    for fp in sorted(not_in_v2)[:10]:
        print(f'  {fp}')
    if len(not_in_v2) > 10:
        print(f'  ... dan {len(not_in_v2)-10} lainnya')

In [ ]:
# ─── Cell 5: Apply Cleaning ke folds_v2 ──────────────────────────────────────
import numpy as np

folds_v3 = folds_v2.copy()
total_before = len(folds_v3)

# Buat lookup dari log
log_indexed = log_all.set_index('filepath')

# --- Relabel dulu (sebelum drop) ---
relabel_mask = folds_v3['filepath'].isin(
    log_indexed[log_indexed['keputusan'] == 'relabel'].index
)
n_relabeled = 0
for idx, row in folds_v3[relabel_mask].iterrows():
    fp = row['filepath']
    if fp in log_indexed.index:
        new_label = log_indexed.loc[fp, 'label_baru']
        if pd.notna(new_label) and str(new_label).strip() != '':
            folds_v3.at[idx, 'label'] = int(float(new_label))
            n_relabeled += 1

print(f'Relabeled : {n_relabeled} gambar')

# --- Drop ---
drop_fps = set(
    log_indexed[log_indexed['keputusan'] == 'drop'].index
) & fps_v2  # hanya drop yang memang ada di folds_v2

folds_v3 = folds_v3[~folds_v3['filepath'].isin(drop_fps)].reset_index(drop=True)
n_dropped = total_before - len(folds_v3) - 0  # relabel tidak mengurangi baris

print(f'Dropped   : {n_dropped} gambar')
print(f'\nTotal sebelum : {total_before:,}')
print(f'Total sesudah : {len(folds_v3):,}')
print(f'Berkurang     : {total_before - len(folds_v3):,} ({(total_before - len(folds_v3))/total_before*100:.2f}%)')

In [ ]:
# ─── Cell 6: Verifikasi Distribusi ───────────────────────────────────────────
CLASS_NAMES = {0: 'Recyclable', 1: 'Electronic', 2: 'Organic'}

print('=' * 55)
print('DISTRIBUSI LABEL — folds_v3')
print('=' * 55)

dist = folds_v3['label'].value_counts().sort_index()
for cls_id, cnt in dist.items():
    print(f'  Kelas {cls_id} ({CLASS_NAMES[cls_id]:12s}): {cnt:6,}')

print(f'  TOTAL                      : {len(folds_v3):6,}')

print('\nDistribusi per fold:')
pivot = folds_v3.groupby(['fold', 'label']).size().unstack(fill_value=0)
pivot.columns = [CLASS_NAMES[c] for c in pivot.columns]
pivot['TOTAL'] = pivot.sum(axis=1)
print(pivot)

# Pastikan tidak ada overlap antar fold
print('\nCek no-overlap antar fold...')
folds_list = sorted(folds_v3['fold'].unique())
all_ok = True
for i in range(len(folds_list)):
    for j in range(i+1, len(folds_list)):
        fps_i = set(folds_v3[folds_v3['fold'] == folds_list[i]]['filepath'])
        fps_j = set(folds_v3[folds_v3['fold'] == folds_list[j]]['filepath'])
        if fps_i & fps_j:
            print(f'❌ LEAKAGE! Fold {folds_list[i]} & {folds_list[j]}')
            all_ok = False
if all_ok:
    print('✅ No-overlap antar fold: KONFIRMASI')

In [ ]:
# ─── Cell 7: Hitung Class Weights ────────────────────────────────────────────
# Formula: weight_i = total / (n_classes * count_i)  — sklearn balanced

n_classes = 3
counts = np.array([dist.get(i, 0) for i in range(n_classes)], dtype=float)
total  = counts.sum()
weights_v3 = total / (n_classes * counts)

print('class_weights_v3:')
for i, w in enumerate(weights_v3):
    print(f'  Kelas {i} ({CLASS_NAMES[i]:12s}): {w:.4f}')

In [ ]:
# ─── Cell 8: Simpan Output ───────────────────────────────────────────────────
import json

FOLDS_V3_PATH   = f'{OUTPUT_DIR}/folds_v3.csv'
WEIGHTS_V3_PATH = f'{OUTPUT_DIR}/class_weights_v3.npy'
SUMMARY_PATH    = f'{OUTPUT_DIR}/cleaning_summary_v3.json'

# Simpan folds_v3.csv
folds_v3.to_csv(FOLDS_V3_PATH, index=False)
print(f'✅ Saved: {FOLDS_V3_PATH}')

# Simpan class_weights_v3.npy
np.save(WEIGHTS_V3_PATH, weights_v3)
print(f'✅ Saved: {WEIGHTS_V3_PATH}')

# Simpan summary JSON
summary = {
    'version'           : 'v3',
    'source_folds'      : 'folds_v2.csv',
    'cleaning_logs'     : [
        'cleaning_log_terisi.csv (review manual — raka)',
        'cleaning_log_misclassified.csv (misclassified review — aga)'
    ],
    'total_before'      : int(total_before),
    'total_after'       : int(len(folds_v3)),
    'n_dropped'         : int(n_dropped),
    'drop_pct'          : round(n_dropped / total_before * 100, 2),
    'n_relabeled'       : int(n_relabeled),
    'class_distribution': {CLASS_NAMES[i]: int(counts[i]) for i in range(n_classes)},
    'class_weights_v3'  : weights_v3.tolist(),
}
with open(SUMMARY_PATH, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'✅ Saved: {SUMMARY_PATH}')

print('\n' + '=' * 55)
print('RINGKASAN CLEANING v3 — Untuk Laporan Panitia')
print('=' * 55)
print(f"Total sebelum cleaning : {summary['total_before']:,}")
print(f"Total sesudah cleaning : {summary['total_after']:,}")
print(f"Dropped                : {summary['n_dropped']:,} ({summary['drop_pct']}%)")
print(f"Relabeled              : {summary['n_relabeled']:,}")
print(f"Class distribution v3  : {summary['class_distribution']}")
print(f"Class weights v3       : {[round(w,4) for w in summary['class_weights_v3']]}")

In [ ]:
# ─── Cell 9: Verifikasi Load Ulang ───────────────────────────────────────────
folds_v3_check   = pd.read_csv(FOLDS_V3_PATH)
weights_v3_check = np.load(WEIGHTS_V3_PATH)

print(f'folds_v3.csv loaded      : {len(folds_v3_check):,} baris ✅')
print(f'class_weights_v3.npy     : {weights_v3_check.round(4).tolist()} ✅')

# Simulasi fold 0
df_train = folds_v3_check[folds_v3_check['fold'] != 0]
df_val   = folds_v3_check[folds_v3_check['fold'] == 0]
print(f'\nFold 0 simulation:')
print(f'  Train: {len(df_train):,} | Val: {len(df_val):,}')
print(f'  Label train: {dict(df_train["label"].value_counts().sort_index())}')
print(f'  Label val  : {dict(df_val["label"].value_counts().sort_index())}')
print('\n✅ Track B siap load folds_v3.csv!')

In [ ]:
# ─── Cell 10: GATE G3 ────────────────────────────────────────────────────────
print('=' * 60)
print('🚦 GATE G3 — Pesan untuk dikirim ke grup:')
print('=' * 60)
print()
print('"GATE G3 HIJAU ✅')
print()
print(f'folds_v3.csv + class_weights_v3.npy tersedia di Drive.')
print(f'Path: [DRIVE_BASE]/output_trackA/')
print()
print(f'Ringkasan cleaning (gabungan 2 log):')
print(f'  - Dropped   : {summary["n_dropped"]} ({summary["drop_pct"]}%)')
print(f'  - Relabeled : {summary["n_relabeled"]}')
print(f'  - Total v3  : {summary["total_after"]:,} sampel')
print()
print('Track B: update path di config.py → folds_v3.csv + class_weights_v3.npy')

---

## ✅ Fase 4 Selesai! GATE G3 Terbuka.

Artefak yang dihasilkan di Drive:
- `folds_v3.csv` — train bersih (gabungan 2 cleaning log)
- `class_weights_v3.npy` — bobot kelas untuk CrossEntropyLoss
- `cleaning_summary_v3.json` — metodologi + angka untuk laporan

**Track B:** Update `config.py` → `folds_csv = '...folds_v3.csv'` + `class_weights_path = '...class_weights_v3.npy'`
